# OME-TIFF vs OME-ZARR

# Download the files from Dandi Archive for a single subject

This repo has code to automatically download data from the CATNIP dataset hosted on dandi archive. Run cell below to download OME-TIFF files. Note, this can take several minutes depending on your internet connection.

In [1]:
import os
import time
from pathlib import Path

import psutil
import tifffile
import zarr

from ome_compare import download_subject_45424_dataset

# download all datasets for subject 45424
# this can take several minutes depending on your internet connection
# files are saved in ./catnip/ by default
# if you want to save the files in a different directory,
# you can pass the root_dir argument
# e.g. download_subject_45424_dataset(root_dir="./my_data")
download_subject_45424_dataset(root_dir=Path("./catnip"))
n4_corrected_filepath: Path = Path(
    r"catnip/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_SPIM.ome.btf")
assert n4_corrected_filepath.exists()


File catnip/derivatives/AtlasLabel/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_space-orig_dseg.ome.btf already exists. Skipping download
File catnip/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_SPIM.ome.btf already exists. Skipping download
File catnip/derivatives/AtlasLabelMasked/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_space-orig_dseg.ome.btf already exists. Skipping download
File catnip/derivatives/FastRadialSymmetryTransform/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_SPIM.ome.btf already exists. Skipping download
File catnip/derivatives/FastRadialSymmetryTransformHemisphere/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_SPIM.ome.btf already exists. Skipping download
File catnip/derivatives/HeatmapsAtlasSpace/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_acq-heatmap_res-25um_SPIM.ome.btf already exists. Skipping download
File catnip/derivatives/FastRadialSymmetryTransformSegmentation/sub-45424flox/micr/sub-45424flox_sam

## Load the file using tifffile

First we will load the entire image into memory using `tifffile`. We will time how long it takes to read the file into memory

In [2]:
# Get the current process
process = psutil.Process(os.getpid())

# Get memory info
memory_info = process.memory_info()
memory_gb = memory_info.rss / (1024**3)  
print(f"Process memory usage (RSS), before loading image: {memory_gb:.2f} GB")
# Load the image and print the time taken
start_time = time.perf_counter()
img = tifffile.imread(n4_corrected_filepath)
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time:.2f} seconds")
memory_info = process.memory_info()
memory_gb = memory_info.rss / (1024**3)  
print(f"Process memory usage (RSS), after loading image: {memory_gb:.2f} GB")
# Calculate size in bytes (number of elements * bytes per element)
size_bytes = img.nbytes
# Convert to GB
size_gb = size_bytes / (1024**3)
print(f"Image array size: {size_gb:.2f} GB")

print(f"Image shape: {img.shape}")


Process memory usage (RSS), before loading image: 0.09 GB
Time taken: 9.00 seconds
Process memory usage (RSS), after loading image: 2.79 GB
Image array size: 11.54 GB
Image shape: (1120, 2560, 2160)


In [3]:
# delete the img variable to free memory
if "img" in locals():
    del img
# Get the current process
process = psutil.Process(os.getpid())

# Get memory info
memory_info = process.memory_info()
memory_gb = memory_info.rss / (1024**3)
print(f"Process memory usage (RSS), before loading zarr store: "
      f"{memory_gb:.2f} GB")
# load the image as a zarr store
start_time = time.perf_counter()
tiff_store = tifffile.imread(n4_corrected_filepath, aszarr=True, mode="r")
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time:.2f} seconds")
zarr_img = zarr.open(tiff_store, mode="r")
# Calculate size in bytes (number of elements * bytes per element)
size_bytes = zarr_img.nbytes # type: ignore
# Convert to GB
size_gb = size_bytes / (1024**3)

# Get memory info
memory_info = process.memory_info()
memory_gb = memory_info.rss / (1024**3)
print("Process memory usage (RSS), after loading zarr store:" 
      f"{memory_gb:.2f} GB")

print(f"Zarr store size: {size_gb:.2f} GB")
if isinstance(zarr_img, zarr.Array):
    print(f"Image shape: {zarr_img.shape}")
else:
    print("Zarr store is a group")


Process memory usage (RSS), before loading zarr store: 0.22 GB
Time taken: 0.09 seconds
Process memory usage (RSS), after loading zarr store:0.23 GB
Zarr store size: 11.54 GB
Image shape: (1120, 2560, 2160)
